# Inference and benchmarking
(on V100 GPUs)

Loads GPTQ models from `../quantisation/_temp/` folder (from `quantisation/project.ipynb`) and benchmarks across thinking budgets / quantisation combinations using vLLMs offline inference.

The CoT-token injection was ported from vLLM 0.12.0 to the Volta compatible 0.7.3 version, since only vLLM ≥ 0.12.0 introduced `--enable-reasoning` and guided-decoding-based think-token injection.
Sadly, this version requires CUDA 12.9, respectively torch 2.9, which drops Volta(sm_70) support entirely.

The behavior is reproduced as `SamplingParams.logits_processors`, injecting `</think>` at thinking-budget boundaries, enabling CoT models(Qwen3) to run on V100(sm_70/cuda12.1).

Given GPU Volta series constraints we must respect:
- `VLLM_ATTENTION_BACKEND=XFORMERS` as only functional attention path,
- `xformers==0.0.27.post2` as last release retaining sm_70 kernels,
- `dtype=float16` since V100 has no `bf16` depth support,
- `quantization=gptq`, `use_marlin=False` since these require sm$\ge$8.0.

We further denote:
- `triton==2.1.0`, last `sm_70`compatible version misses `default_dump_dir`
- `default_override_dir` since `triton>=2.2.0`
- `vLLM 0.7.3`: lazy `triton_utils.custom_cache_manager` loading at quant config uses, aliased to `default_cache_dir` to not interfere with `sm_70` paths

- Datasets are loaded from huggingface

In [ ]:
import os
os.environ["VLLM_ATTENTION_BACKEND"] = "XFORMERS" # SM70

import triton.runtime.cache as _triton_cache
if not hasattr(_triton_cache, "default_dump_dir"): _triton_cache.default_dump_dir = _triton_cache.default_cache_dir
if not hasattr(_triton_cache, "default_override_dir"): _triton_cache.default_override_dir = _triton_cache.default_cache_dir

In [ ]:
import os

# CROSS VAL
DATASET_SPLIT = int(os.environ.get("DATASET_SPLIT", "0"))
CROSS_VAL     = os.environ.get("CROSS_VAL", "0") == "1"

# SAMPLING (smoke (5) > subset (N) > full (None))
_subset_env = os.environ.get("SUBSET_N", "")
SMOKE_TEST_N_SAMPLES = (
    5           if os.environ.get("SMOKE_TEST", "1") != "0"
    else int(_subset_env) if _subset_env
    else None
)

# PATH
BASE_PATH = "../quantisation/_temp"
PRED_LOG_PATH = os.path.abspath("prediction_log.csv")
HF_DATASETS = [
    {
        "name":   "hendrycks_math",
        "handle": "HuggingFaceH4/MATH-500",
        "config": None,
        "split":  "test",
        "equiv":  "math",
    },
    {
        "name":   "drop",
        "handle": "ucinlp/drop",
        "config": None,
        "split":  "validation",
        "equiv":  "drop",
    },
]

# MODELS
MODEL_SELECTION         = ["qwen", "deepseek"]
QUANTISATION_BITS       = [4, 8]
QUANTISATION_GROUPLEVEL = 128 # for VRAM fixed
THINKING_BUDGETS        = [64, 128, 256, 512]
ANSWER_TOKENS           = 256 # constant
# alternative prompt (first run) with better performance: 'Place your precise answer in <result>value</result>'
INFERENCE_PROMPT        = "State your final answer as value in <result></result> tags."
handle_map = {
    "qwen":      "Qwen/Qwen3-0.6B",
    "deepseek":  "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
}

In [ ]:
import itertools, logging
from scripts.benchmark import load_benchmark_datasets

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.FileHandler("inf_run.log"),
        logging.StreamHandler(),
    ],
    force=True,
)

logging.info(f"[DATA] Loading benchmark datasets (split={DATASET_SPLIT}, smoke={SMOKE_TEST_N_SAMPLES})")
all_datasets = load_benchmark_datasets(HF_DATASETS, DATASET_SPLIT, SMOKE_TEST_N_SAMPLES, INFERENCE_PROMPT)
logging.info(f"[DATA] {sum(len(d.prompts) for d in all_datasets)} samples loaded "
             f"({sum(d.full_size for d in all_datasets)} full)")

In [ ]:
import itertools, json, logging, os, pathlib, subprocess, time

_worker_path = pathlib.Path(__file__).parent / "_worker.py" if "__file__" in dir() else pathlib.Path("_worker.py")
inf_dir = _worker_path.parent.resolve()

# SUMAMRY

params     = list(itertools.product(MODEL_SELECTION, QUANTISATION_BITS))
n_configs  = len(params)
n_budgets  = len(THINKING_BUDGETS)
n_datasets = len(all_datasets)
total_runs = n_configs * n_budgets * n_datasets

print(f"\n{'='*60}")
print(f"PREFLIGHT")
print(f"  Model configs : {n_configs}  ({', '.join(f'{m} {b}b' for m,b in params)})")
print(f"  Budgets       : {n_budgets}  {THINKING_BUDGETS}")
print(f"  Datasets      :")
for d in all_datasets:
    smoke_note = f"  <- smoke {len(d.prompts)}/{d.full_size}" if SMOKE_TEST_N_SAMPLES else ""
    print(f"    {d.name:30s}  {d.full_size} samples{smoke_note}")
print(f"  Total runs    : {n_configs} x {n_budgets} x {n_datasets} = {total_runs}")
print(f"  Note: each model config runs in a subprocess (GPU freed on exit)")
print(f"{'='*60}\n")


# BENCHMARKING AND INFERENCE

for model_name, quant_bits in params:
    model_path = os.path.abspath(os.path.join(
        BASE_PATH, f"{model_name}_Q_{quant_bits}B{QUANTISATION_GROUPLEVEL}G"
    ))
    if not os.path.exists(model_path):
        logging.warning(f"[SKIP] {model_path} not found")
        continue

    cfg = {
        "model_name":             model_name,
        "quant_bits":             quant_bits,
        "quant_groupsize":        QUANTISATION_GROUPLEVEL,
        "handle":                 handle_map[model_name],
        "base_path":              os.path.abspath(BASE_PATH),
        "thinking_budgets":       THINKING_BUDGETS,
        "answer_tokens":          ANSWER_TOKENS,
        "dataset_split":          DATASET_SPLIT,
        "smoke_n":                SMOKE_TEST_N_SAMPLES,
        "hf_datasets":            HF_DATASETS,
        "inference_prompt":       INFERENCE_PROMPT,
        "pred_log_path":          PRED_LOG_PATH,
        "benchmark_results_path": str(inf_dir / "benchmark_results.csv"),
        "log_path":               str(inf_dir / f"inf_{model_name}_{quant_bits}b.log"),
    }

    cfg_file = inf_dir / f"_cfg_{model_name}_{quant_bits}b.json"
    cfg_file.write_text(json.dumps(cfg, indent=2))

    cmd = ["uv", "run", "python", str(_worker_path), "--config", str(cfg_file)]
    logging.info(f"[RUN] Launching subprocess: {model_name} {quant_bits}b")
    print(f"\n{'='*60}\n[SUBPROCESS] {model_name} {quant_bits}b\n{'='*60}")

    t0 = time.perf_counter()
    proc = subprocess.run(cmd, cwd=str(inf_dir))
    elapsed = time.perf_counter() - t0
    cfg_file.unlink(missing_ok=True)

    if proc.returncode != 0:
        raise RuntimeError(
            f"Worker failed for {model_name} {quant_bits}b (exit {proc.returncode}). "
            f"See {cfg['log_path']}"
        )
    logging.info(f"[RUN] {model_name} {quant_bits}b completed in {elapsed:.1f}s")

logging.info("[RUN] All configurations complete")